## Question 1:

**Input**

| a | b |
|---|---|
| 1 | 5 |
| 1 | 5 |
| 1 | 5 |
| 2 | 6 |
| 1 | 5 |

**Expected Output**

| a | sum_b |
|---|-------|
| 1 | 20    |

In [0]:
data = [
    (1, 5),
    (1, 5),
    (1, 5),
    (2, 6),
    (1, 5)
]

df = spark.createDataFrame(
    data,
    ["a", "b"]
)

df.show()

df.createOrReplaceTempView('table_name')

+---+---+
|  a|  b|
+---+---+
|  1|  5|
|  1|  5|
|  1|  5|
|  2|  6|
|  1|  5|
+---+---+



In [0]:
%sql
select a, sum(b) as sum_b from table_name group by a having sum(b) > 10

a,sum_b
1,20


In [0]:
from pyspark.sql import functions as F
df = df.groupBy(F.col('a'))\
        .agg(F.sum(F.col('b')).alias('sum_b'))\
        .filter(F.col('sum_b') > 10)


df.display()

a,sum_b
1,20


In [0]:
import pandas as pd

data = {"a": [1,1,1,2,1], "b":[5,5,5,6,5]}

df = pd.DataFrame(data)

result = (
  df.groupby("a", as_index=False)["b"]
    .sum()
    .rename(columns={"b":"sum_b"})
    .query("sum_b > 10")
)

print(result)

   a  sum_b
0  1     20


Question 2:


| Brand_1 | Brand_2 | Brand_3 | Winner |
|---------|---------|---------|--------|
| A       | B       | C       | B      |
| B       | C       | E       | E      |
| C       | A       | D       | D      |
| D       | E       | A       | A      |
| F       | B       | C       | F      |



Expected Output


| Brand_Name | Total_Appearances | Wins | Losses |
|------------|-------------------|-----|-------|
| A          | 3                 | 1   | 2     |
| B          | 3                 | 1   | 2     |
| C          | 4                 | 0   | 4     |
| D          | 2                 | 1   | 1     |
| E          | 2                 | 1   | 1     |
| F          | 1                 | 1   | 0     |


In [0]:

data = [
    ("A", "B", "C", "B"),
    ("B", "C", "E", "E"),
    ("C", "A", "D", "D"),
    ("D", "E", "A", "A"),
    ("F", "B", "C", "F")
]

df = spark.createDataFrame(
    data,
    ["Brand_1", "Brand_2", "Brand_3", "Winner"]
)

df.createOrReplaceTempView("table_name")

In [0]:
%sql
with cte1 as
(
select Brand_1 as Brand_Name, case when Brand_1 = Winner then 1 else 0 end as Winner from table_name
union all
select Brand_2 as Brand_Name, case when Brand_2 = Winner then 1 else 0 end as Winner from table_name
union all
select Brand_3 as Brand_Name, case when Brand_3 = Winner then 1 else 0 end as Winner from table_name
)
select Brand_Name, count(Brand_Name) as Total_Appearances, sum(Winner) as Wins, (count(Brand_Name) - sum(Winner)) as Losses from cte1 group by Brand_Name

Brand_Name,Total_Appearances,Wins,Losses
A,3,1,2
B,3,1,2
C,4,0,4
D,2,1,1
F,1,1,0
E,2,1,1


In [0]:
from pyspark.sql import functions as F
df1 = df.withColumn('Brand_Name', F.col('Brand_1'))\
		.withColumn('Winner', F.when(F.col('Brand_1') == F.col('Winner'), 1).otherwise(0))\
		.unionByName(
			df.withColumn('Brand_Name', F.col('Brand_2'))\
			.withColumn('Winner', F.when(F.col('Brand_2') == F.col('Winner'), 1).otherwise(0))\
			.unionByName(
				df.withColumn('Brand_Name', F.col('Brand_3'))\
				.withColumn('Winner', F.when(F.col('Brand_3') == F.col('Winner'), 1).otherwise(0))
			)
		)\
		.groupBy(F.col('Brand_Name'))\
		.agg(F.count('Brand_Name').alias('Total_Appearances'), F.sum(F.col('Winner')).alias('Winner'), (F.count('Brand_Name') - F.sum('Winner')).alias('Losses'))\
		.select('Brand_Name', 'Total_Appearances', 'Winner', 'Losses')
  
df1.display()

Brand_Name,Total_Appearances,Winner,Losses
A,3,1,2
B,3,1,2
C,4,0,4
D,2,1,1
F,1,1,0
E,2,1,1


In [0]:
import pandas as pd
import numpy as np

pdf = df.toPandas()

df_b1 = pdf.assign(
    Brand_Name=pdf["Brand_1"], Winner=np.where(pdf["Brand_1"] == pdf["Winner"], 1, 0)
)

df_b2 = pdf.assign(
    Brand_Name=pdf["Brand_2"], Winner=np.where(pdf["Brand_2"] == pdf["Winner"], 1, 0)
)

df_b3 = pdf.assign(
    Brand_Name=pdf["Brand_3"], Winner=np.where(pdf["Brand_3"] == pdf["Winner"], 1, 0)
)

df1 = pd.concat([df_b1, df_b2, df_b3], ignore_index=True)

result = (
    df1.groupby("Brand_Name")
    .agg(Total_Appearances=("Brand_Name", "count"), Winner=("Winner", "sum"))
    .reset_index()
)

result["Losses"] = result["Total_Appearances"] - result["Winner"]

print(result)

  Brand_Name  Total_Appearances  Winner  Losses
0          A                  3       1       2
1          B                  3       1       2
2          C                  4       0       4
3          D                  2       1       1
4          E                  2       1       1
5          F                  1       1       0


## Question 3

**Input Table**

| Team_1 | Team_2 | Winner |
|--------|--------|--------|
| India  | SL     | India  |
| SL     | Aus    | Aus    |
| SA     | Eng    | Eng    |
| Eng    | NZ     | NZ     |
| Aus    | India  | India  |

**Expected Output**

| Team_Name | Matches_played | no_of_wins | no_of_losses |
|-----------|---------------|------------|--------------|
| India     | 2             | 2          | 0            |
| SL        | 2             | 0          | 2            |
| Aus       | 2             | 1          | 1            |
| SA        | 1             | 0          | 1            |
| Eng       | 2             | 1          | 1            |
| NZ        | 1             | 1          | 0            |